In [ ]:
# prompt: connect to drive

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install owlready2
from owlready2 import *
!pip install rdflib
!pip install networkx
import rdflib
import networkx as nx
!pip install jpype1
import jpype
from re import sub
import re

# Load OWL/RDF data
def create_gephi_graph(file_path,file_name):
  g = rdflib.Graph()
  g.parse(file_path)

# Create a directed graph
  nx_graph = nx.DiGraph()

# Extract triples and add them as edges
  for subj, pred, obj in g:
    nx_graph.add_edge(str(subj), str(obj), label=str(pred))

# Export to GraphML
  return nx.write_gexf(nx_graph, f"{file_name}.gexf")

ofl = get_namespace("https://purl.bioontology.org/ontology/OFL/")
fma= get_namespace("http://purl.org/sig/ont/fma/")
fma_obo= get_namespace("http://purl.obolibrary.org/obo/FMA_")
bspo= get_namespace("http://purl.obolibrary.org/obo/BSPO_")
obo = get_namespace("http://purl.obolibrary.org/obo/")
nico= get_namespace("https://purl.bioontology.org/ontology/OFL-Nicoedit/")
nico2= get_namespace("https://purl.bioontology.org/ontology/OFL-Nicoedit#")

# Correct IRI prefix
correct_prefix = "https://purl.bioontology.org/ontology/OFL/"
correct_prefix_fma = "http://purl.obolibrary.org/obo/FMA_"

def camel_case_to_label(name):
    import re
    # Insert spaces before capital letters that are preceded by lowercase letters
    separated_words = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', name)
    # Capitalize the first letter and lowercase the rest
    return separated_words[0].upper() + separated_words[1:].lower()

# Counter for generating unique IDs
id_counter = 10000  # Start from an arbitrary number

# Function to generate unique IDs
def generate_id():
    global id_counter
    id_counter += 1
    return f"OFLID{id_counter}"

def update_relations_for_subclasses(cls):
    # Collect restrictions to remove
    restrictions_to_remove = []
    restrictions_to_add = []

    # Check if the class has any restrictions
    if cls.is_a:
        for restriction in cls.is_a:
            if isinstance(restriction, owlready2.Restriction):  # Check if it’s a restriction
                if isinstance(restriction.property, owlready2.ObjectPropertyClass):  # Ensure it's an object property
                    # Check if the restriction involves "arterial supply"
                    if restriction.property.iri == "http://purl.org/sig/ont/fma/arterial_supply":
                        # Create the new restriction
                        new_restriction = arterial_flap_supply_prop.some(restriction.value)
                        restrictions_to_remove.append(restriction)
                        restrictions_to_add.append(new_restriction)
                        print(f"Class: {cls.name}, Replaced: {restriction.property} with {new_restriction.property}")

    # Now remove the old restrictions and append the new ones
    for restriction in restrictions_to_remove:
        cls.is_a.remove(restriction)

    for new_restriction in restrictions_to_add:
        cls.is_a.append(new_restriction)

def update_venous_for_subclasses(cls):
    # Collect restrictions to remove
    restrictions_to_remove = []
    restrictions_to_add = []

    # Check if the class has any restrictions
    if cls.is_a:
        for restriction in cls.is_a:
            if isinstance(restriction, owlready2.Restriction):  # Check if it’s a restriction
                if isinstance(restriction.property, owlready2.ObjectPropertyClass):  # Ensure it's an object property
                    # Check if the restriction involves "venous drainage" or "drains into"
                    if restriction.property.iri == "http://purl.org/sig/ont/fma/drains_into" or restriction.property.iri == "http://purl.org/sig/ont/fma/venous_drainage":
                        # Create the new restriction
                        new_restriction = flap_onto_2.world["https://purl.bioontology.org/ontology/ONTOFLAP/OFLID10273"].some(restriction.value)
                        restrictions_to_remove.append(restriction)
                        restrictions_to_add.append(new_restriction)
                        print(f"Class: {cls.name}, Replaced: {restriction.property} with {new_restriction.property}")

    # Now remove the old restrictions and append the new ones
    for restriction in restrictions_to_remove:
        cls.is_a.remove(restriction)

    for new_restriction in restrictions_to_add:
        cls.is_a.append(new_restriction)

def update_nerval_for_subclasses(cls):
    # Collect restrictions to remove
    restrictions_to_remove = []
    restrictions_to_add = []

    # Check if the class has any restrictions
    if cls.is_a:
        for restriction in cls.is_a:
            if isinstance(restriction, owlready2.Restriction):  # Check if it’s a restriction
                if isinstance(restriction.property, owlready2.ObjectPropertyClass):  # Ensure it's an object property
                    # Check if the restriction involves "venous drainage" or "drains into"
                    if restriction.property.iri == "http://purl.org/sig/ont/fma/nerve_supply":
                        # Create the new restriction
                        new_restriction = flap_onto_2.world["https://purl.bioontology.org/ontology/ONTOFLAP/OFLID10289"].some(restriction.value)
                        restrictions_to_remove.append(restriction)
                        restrictions_to_add.append(new_restriction)
                        print(f"Class: {cls.name}, Replaced: {restriction.property} with {new_restriction.property}")

    # Now remove the old restrictions and append the new ones
    for restriction in restrictions_to_remove:
        cls.is_a.remove(restriction)

    for new_restriction in restrictions_to_add:
        cls.is_a.append(new_restriction)

def update_property(my_onto, enti, OLD_PROPERTY_IRI, NEW_PROPERTY_IRI ):
    # === Define the IRIs of the property to replace ===

    # Prepare lists to track changes (optional)
    restrictions_to_remove = []
    restrictions_to_add = []
    new_property=my_onto.world[f"{NEW_PROPERTY_IRI}"]
    # Iterate over restrictions

    for restriction in enti.get_properties():
        if isinstance(restriction, owlready2.Restriction):  # Check if it’s a restriction
          if isinstance(restriction.property, owlready2.ObjectPropertyClass):  # Ensure it's an object property
            if restriction.property.iri == OLD_PROPERTY_IRI:
                restrictions_to_remove.append(restriction)
                if enti in list(my_onto.classes()):
                # Rebuild restriction with new property
                  if restriction.type == SOME:
                    new_restriction = new_property.some(restriction.value)
                  elif restriction.type == ONLY:
                    new_restriction = new_property.only(restriction.value)
                  elif restriction.type == EXACTLY:
                    new_restriction = new_property.exactly(restriction.cardinality, restriction.value)
                  elif restriction.type == MIN:
                    new_restriction = new_property.min(restriction.cardinality, restriction.value)
                  elif restriction.type == MAX:
                    new_restriction = new_property.max(restriction.cardinality, restriction.value)
                  else:
                    print(f"Skipping unknown restriction type in {enti.name}: {restriction}")
                    continue
                elif enti in list(my_onto.individuals()):
                   new_restriction = new_property(restriction.value)
                restrictions_to_add.append(new_restriction)

    # Apply changes
    for r in restrictions_to_remove:
        enti.get_properties.remove(r)
    for r in restrictions_to_add:
        enti.is_a.append(r)

    if restrictions_to_remove:
        print(f"Updated {enti.name}: replaced {len(restrictions_to_remove)} restriction(s) using '{OLD_PROPERTY_IRI}'.")


# Function to merge classes with the same label
def merge_classes_by_label(onto, label):
    # Find all classes with the given label
    classes_to_merge = [cls for cls in onto.classes() if label == cls.label]

    if len(classes_to_merge) <= 1:
        pass
    else:
    # Pick the first class as the "main" class to retain
      main_class = classes_to_merge[0]

      for cls in classes_to_merge[1:]:
        # Merge restrictions (is_a) and other properties from each class
        for restriction in cls.is_a:
            # Add restrictions to the main class
            if restriction not in main_class.is_a:
                main_class.is_a.append(restriction)

        # Remove the redundant class
        print(f"Merging {cls.name} into {main_class.name}")
        destroy_entity(cls)  # Remove the duplicate class

# Shut down JVM if it's already running
if jpype.isJVMStarted():
    jpype.shutdownJVM()

# Start JVM with 24GB heap
jpype.startJVM(jpype.getDefaultJVMPath(), "-Xmx11g")


OSError: JVM cannot be restarted

In [ ]:
# prompt: import an ontology using olwready
# fma_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma.owl").load()
# bspo_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/bspo-full.owl").load()
flap_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_13_1.owl").load()

In [ ]:
entities=list(flap_onto.classes()) + list(flap_onto.object_properties()) + list(flap_onto.data_properties()) + list(flap_onto.individuals())

NameError: name 'flap_onto' is not defined

In [ ]:

# Process all entities: classes, object properties, data properties, and individuals
for entity in entities:
    if  not any(exclude in entity.iri for exclude in ["fma", "/obo/"]):
        if not entity.iri.startswith(correct_prefix):
            # Generate a new IRI
            new_iri = f"{correct_prefix}{entity.name}"
            # Update the entity's IRI and assign the label
            try:
              entity.iri = new_iri
            except:
              print("Prefix change failed: ", entity.iri, new_iri)


In [ ]:
for entity in entities:
    if  not any(exclude in entity.iri for exclude in ["fma", "/obo/"]):
        if not entity.name.startswith("OFLID"):
            # Create a human-readable label
            new_comment = camel_case_to_label(entity.name.replace("_", " "))
            # Update IRI and label
            while True:
              try:
                entity.name = generate_id()
                break
              except:
                generate_id
            #entity.comment = [new_comment]
            try:
              new_iri = f"{correct_prefix}{entity.name}"
            except:
              print(entity.name)
            # Update the entity's IRI and assign the label
            entity.iri = new_iri
            # Output changes for verification
            print(f"Updated: {entity.name} -> IRI: {new_iri}, Label: {entity.label}")

Updated: OFLID10101 -> IRI: https://purl.bioontology.org/ontology/OFL/OFLID10101, Label: [locstr('Subcutaneous adipose tissue of Left axilla (FMA:45303)', 'de')]
Updated: OFLID10118 -> IRI: https://purl.bioontology.org/ontology/OFL/OFLID10118, Label: [locstr('Subcutaneous adipose tissue of Axilla (FMA:24864)', 'de')]
Updated: OFLID10119 -> IRI: https://purl.bioontology.org/ontology/OFL/OFLID10119, Label: [locstr('Subcutaneous adipose tissue of Right axilla (FMA:45302)', 'de')]
Updated: OFLID10122 -> IRI: https://purl.bioontology.org/ontology/OFL/OFLID10122, Label: [locstr('Subcutaneous adipose tissue of Right labium majus (FMA:20369)', 'de')]
Updated: OFLID10124 -> IRI: https://purl.bioontology.org/ontology/OFL/OFLID10124, Label: [locstr('Subcutaneous adipose tissue of Labium majus (FMA:20367)', 'de')]
Updated: OFLID10125 -> IRI: https://purl.bioontology.org/ontology/OFL/OFLID10125, Label: [locstr('Subcutaneous adipose tissue of Left labium majus (FMA:20372)', 'de')]
Updated: OFLID1015

In [ ]:
# Iterate through all entities in the ontology
for entity in entities:
    # Check if the entity has a label and it's a list
    if entity.label and isinstance(entity.label, list):
        cleaned_labels = []
        for label in entity.label:
            # Convert locstr to string if necessary
            label_str = str(label)
            # Remove "['"
            cleaned_label = label_str.replace("['", "")
            # Remove "']"
            cleaned_label = cleaned_label.replace("']", "")
            cleaned_label = cleaned_label.replace("'", "")
            cleaned_label = cleaned_label.replace('"', "")
            cleaned_labels.append(cleaned_label)
        # Update the entity's label with the cleaned list
        entity.label = cleaned_labels

In [ ]:
for ent in entities:
  if ent.iri.startswith(correct_prefix):
    if "pedical" in ent.label.first():
      ent.label[0]=sub("pedical","pedicle",ent.label[0] )
      print(ent.label)
for ent in entities:
  if ent.iri.startswith(correct_prefix):
    if "pedicle " in ent.label.first() and not ("pedicle of the" in ent.label.first() ):
      ent.label[0]=sub("pedicle","pedicle of the",ent.label[0])
      if "flap" in ent.label.first() and not (" flap" in ent.label.first() ):
        ent.label[0]=sub("flap"," flap",ent.label[0] )
      print(ent.label)

In [ ]:

flap_onto.save(file="/content/drive/MyDrive/flap_onto_corrected_2.owl")

# Flap onto 2 from here : IRIs and names and labels correct until here

In [ ]:
# Load your ontology
fma_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma.owl").load()
flap_onto_2 = get_ontology("file:///content/drive/MyDrive/flap_onto_corrected_2.owl").load()
flap_onto_2.imported_ontologies.append(fma_onto)


In [ ]:
# Get the Flaps class
Flaps = flap_onto_2.world["https://purl.bioontology.org/ontology/ONTOFLAP/OFLID10002"]
print(Flaps)
# Find properties based on labels
arterial_supply_prop = flap_onto_2.world["http://purl.org/sig/ont/fma/arterial_supply"]

arterial_flap_supply_prop = flap_onto_2.world["https://purl.bioontology.org/ontology/ONTOFLAP/OFLID10259"]



In [ ]:
list_of_oflid_classes=[i for i in flap_onto_2.classes() if "fma" not in i.name]

for i in list_of_oflid_classes:
  update_relations_for_subclasses(i)
  update_venous_for_subclasses(i)
  update_nerval_for_subclasses(i)
  for k in i.is_a:
    if isinstance(k, owlready2.class_construct.Restriction):
      if isinstance(k.property, owlready2.ObjectPropertyClass):  # Ensure it's an object property
            if k.property.iri=="https://purl.bioontology.org/ontology/ONTOFLAP/OFLID10259":
               print(f"Class: {i.label}, Property: {k.property.label}, Value: {k.value}")

In [ ]:
flap_onto_2.save(file="/content/drive/MyDrive/flap_onto_corrected_3.owl")

# Flap onto_4 from here

In [ ]:
oflid_labels=[i.label for i in list_of_oflid_classes]

for i in oflid_labels:
     merge_classes_by_label(flap_onto_4, i)

In [ ]:
fma_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma.owl").load()
flap_onto_4 = get_ontology("file:///content/drive/MyDrive/flap_onto_corrected_4.owl").load()
flap_onto_4.imported_ontologies.append(fma_onto)

In [ ]:
list_of_oflid_classes=[i for i in flap_onto_4.classes() if "fma" not in i.name]

for i in list_of_oflid_classes:
  update_relations_for_subclasses(i)
  update_venous_for_subclasses(i)
  update_nerval_for_subclasses(i)

In [ ]:
sync_reasoner([flap_onto_4])

In [ ]:
flap_onto_4.save(file="/content/drive/MyDrive/flap_onto_corrected_5.owl")

# Flap onto_6 from here

In [ ]:
fma_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma.owl").load()
flap_onto_6 = get_ontology("file:///content/drive/MyDrive/flap_onto_corrected_6.owl").load()
flap_onto_6.imported_ontologies.append(fma_onto)

In [ ]:
with flap_onto_6:
    sync_reasoner([flap_onto_6])

In [ ]:
inferences = get_ontology("http://test.org/onto_inferences.owl")
with inferences:
    sync_reasoner([flap_onto_6])

In [ ]:
print(list(inferences.inconsistent_classes().label))

In [ ]:
ofl=list(flap_onto_6.inconsistent_classes())

In [ ]:
print([i for i in ofl])

In [ ]:
flap_onto_6.save(file="/content/drive/MyDrive/flap_onto_corrected_7.owl")

Flap_onto_8 from here

In [ ]:
#flap_onto_corrected_7 was reedited in protege, than saved as flap_onto_8 and loaded:

flap_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_8.rdf").load()

In [ ]:
id_counter = 10500

In [ ]:
#non-pythonic and lazy copy and paste from above

for entity in list(flap_onto.classes()) + list(flap_onto.object_properties()) + list(flap_onto.data_properties()) + list(flap_onto.individuals()):
    if  not any(exclude in entity.iri for exclude in ["fma", "/obo/"]):
        if not entity.iri.startswith(correct_prefix):
            # Generate a new IRI
            new_iri = f"{correct_prefix}{entity.name}"
            # Update the entity's IRI and assign the label
            entity.iri = new_iri
        if not entity.name.startswith("OFLID"):
            # Create a human-readable label
            new_label = camel_case_to_label(entity.name.replace("_", " "))
            # Update IRI and label
            entity.label = [new_label]
            entity.name = generate_id()
            new_iri = f"{correct_prefix}{entity.name}"
            # Update the entity's IRI and assign the label
            entity.iri = new_iri
            # Output changes for verification
            print(f"Updated: {entity.name} -> IRI: {new_iri}, Label: {new_label}")

Updated: OFLID11152 -> IRI: https://purl.bioontology.org/ontology/OFL/OFLID11152, Label: Vein from gastrocnemius muscle
Updated: OFLID11153 -> IRI: https://purl.bioontology.org/ontology/OFL/OFLID11153, Label: Artery to soleus muscle
Updated: OFLID11154 -> IRI: https://purl.bioontology.org/ontology/OFL/OFLID11154, Label: Vein to soleus muscle
Updated: OFLID11155 -> IRI: https://purl.bioontology.org/ontology/OFL/OFLID11155, Label: Second to fifth metacarpal artery


In [ ]:
flap_onto.save(file="/content/drive/MyDrive/flap_onto_corrected_9.owl")

Flap onto 10 Nico edit from here

In [ ]:
world=World()

In [ ]:
fma_onto = world.get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma.owl").load()
bspo_onto = world.get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/bspo-full.owl").load()

In [ ]:
flap_onto_10 = world.get_ontology("/content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/Onto 10 Nico edit.rdf").load()

In [ ]:
target_iri="https://purl.bioontology.org/ontology/OFL-Nicoedit/OFLID10259"

entity_list=world.search(iri=target_iri)

target_property=entity_list[0]

In [ ]:

# Replace "http://test.org/onto.owl#has_age" with the full URI of the property
# you want to search for from your own ontology.
# You can get the URI of a property like this: print(your_property.iri)
art_supply = target_property

# Now you can get its .iri attribute
print(art_supply)

# This is the SPARQL query.
# It selects all subjects, predicates, and objects
# where the predicate matches the specific URI you provide.
query = f"""
    SELECT ?subject ?predicate ?object
    WHERE {{
        ?subject <{target_property.iri}> ?object .
      BIND(<{target_property.iri}> AS ?predicate)
    }}
"""
# Execute the query on your ontology's world
# In your case, you would use:
results = list(flap_onto_10.world.sparql(query))
results = list(flap_onto_10.world.sparql(query))


# --- Display the results in a table-like format ---

if not results:
    print(f"No triples found with the property <{target_property.name}>")
else:
    # Print table header
    print(f"{'Subject':<40} | {'Predicate':<40} | {'Object':<40}")
    print("-" * 124)

    # Print each row
    for row in results:
        # Each 'item' in the row can be an entity (class, individual) or a literal (string, number).
        # We check the type to print it nicely.
        subject_str = row[0].name if hasattr(row[0], 'name') else str(row[0])
        predicate_str = row[1].name if hasattr(row[1], 'name') else str(row[1])
        object_str = row[2].name if hasattr(row[2], 'name') else str(row[2])

        print(f"{subject_str:<40} | {predicate_str:<40} | {object_str:<40}")


In [ ]:
property_object = flap_onto_10["OFLID1000069"]

# Then you could get its IRI (which you already know)
property_uri = property_object.iri

In [ ]:
for p in flap_onto_10.object_properties():
    print(p.name)

In [ ]:
# --- Diagnostic Code ---

# 1. Define the short name of the property you are looking for.
property_name = "OFLID10259"

# 2. Try to get the property object from your loaded ontology.
target_property = flap_onto_10[property_name]

# 3. Check if the property was found and print its IRI.
if not target_property:
    print(f"CRITICAL ERROR: Could not find property '{property_name}' in the ontology.")
else:
    # This is the most important line. Let's see what the IRI is.
    property_uri = target_property.iri
    print(f"Found property object: {target_property}")
    print(f"The IRI for this property is: {property_uri}") # <-- CHECK THIS OUTPUT

    # 4. Build the query using the IRI we just found.
    query = f"""
        SELECT ?subject ?object
        WHERE {{
            ?subject <{property_uri}> ?object .
        }}
    """
    print("\nFinal SPARQL query being executed:")
    print(query) # <-- CHECK THIS OUTPUT

    # 5. Execute the query
    results = list(flap_onto_10.world.sparql(query))

    if not results:
        print(f"\nQuery ran successfully but found 0 results for <{property_uri}>.")
    else:
        print(f"\nSuccess! Found {len(results)} results.")
        # ... (code to print results)

In [ ]:
property_to_find=target_property



if not property_to_find:
    print(f"ERROR: Could not find property '{property_name}'.")
else:
    property_iri = property_to_find.iri
    print(f"Found property '{property_name}' with IRI: <{property_iri}>")

    # --- Step 2: Build the SPARQL query to find class definitions AND their labels ---
    query = f"""
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX owl: <http://www.w3.org/2002/07/owl#>

        SELECT ?defined_class ?class_label ?supplied_by_class ?filler_label
        WHERE {{
          ?defined_class rdfs:subClassOf ?restriction .

          ?restriction a owl:Restriction ;
                       owl:onProperty <{property_iri}> ;
                       owl:someValuesFrom ?supplied_by_class .

          OPTIONAL {{ ?defined_class rdfs:label ?class_label . }}
          OPTIONAL {{ ?supplied_by_class rdfs:label ?filler_label . }}
        }}
    """
    print("\nExecuting query to find class definitions and labels...")

    # --- Step 3: Execute and display the results ---
    results = list(flap_onto_10.world.sparql(query))

    if not results:
        print("\nNo classes found with that specific subclass definition.")
    else:
        print(f"\nSuccess! Found {len(results)} matching class definitions:")
        print(f"{'Defined Class (SubClassOf)':<60} | {'Is Supplied By (someValuesFrom)':<60}")
        print("-" * 121)

        for row in results:
            # --- Handle the Defined Class (SubClassOf) ---
            class_entity = row[0]
            class_label = row[1]
            class_display = class_label if class_label is not None else class_entity.name

            # --- Handle the Filler Class (someValuesFrom) ---
            filler_entity = row[2]
            filler_label = row[3]

            filler_display = ""
            # Check if a label exists for the filler
            if filler_label is not None:
                filler_display = filler_label
            # NEW: Check if the filler is a logical 'Or' (UnionOf) expression
            elif isinstance(filler_entity, Or):
                # Build a string like "ClassA or ClassB" from the components
                class_names = [c.name for c in filler_entity.Classes]
                filler_display = " or ".join(class_names)
            # Otherwise, fall back to the entity's name
            elif hasattr(filler_entity, 'name'):
                filler_display = filler_entity.name
            # Final fallback to string representation
            else:
                filler_display = str(filler_entity)

            print(f"{str(class_display):<60} | {str(filler_display):<60}")



Flap onto 11 created here

In [ ]:
flap_onto_11=flap_onto

In [ ]:
for ent in entities:
  if "ONTOFLAP" in ent.name:
    print(ent.iri)

In [ ]:
flap_onto_11.save(file="/content/drive/MyDrive/flap_onto_corrected_11.owl")

In [ ]:
create_gephi_graph("/content/drive/MyDrive/flap_onto_corrected_11.owl","flap_onto_corrected_11")

In [ ]:
print(ofl.OFLID10005.label)


['Flap pedicle of the deep inferior epigastric perforator flap']


In [ ]:
# Assuming flap_onto is your currently loaded ontology object
nico_entities = [entity for entity in list(flap_onto.classes()) + list(flap_onto.object_properties()) + list(flap_onto.data_properties()) + list(flap_onto.individuals()) if entity.iri.startswith("https://purl.bioontology.org/ontology/OFL-Nicoedit/")]

print(f"Found {len(nico_entities)} entities with the OFL-Nicoedit prefix:")
for entity in nico_entities:
    print(f"- {entity.iri}")
    if entity.label:
        print(f"  Label: {entity.label}")

Found 0 entities with the OFL-Nicoedit prefix:


In [ ]:
# Assuming you have a class object, for example:
# my_class = flap_onto["YourClassName"] # Replace "YourClassName" with the actual class name

# Example using a known class from your ontology (replace with a class you want to inspect)
# Let's use OFLID10005 as an example
my_class = flap_onto.world["https://purl.bioontology.org/ontology/OFL/OFLID10148"]


print(f"Superclasses of {my_class.label.first() or my_class.name}:")
for superclass in my_class.is_a:
    if isinstance(superclass, owlready2.Restriction):
        print(f"- Anonymous Superclass (Restriction): {superclass}")
    else:
        print(f"- Named Superclass: {superclass.label.first() or superclass.name}")

Superclasses of Auricularis posterior flap:
- Named Superclass: Flaps originating from the head
- Anonymous Superclass (Restriction): fma.part.some(fma.fma322630 | fma.fma46857 | fma.fma61013 | fma.fma74315 | fma.fma76863)


Flap onto corrected 13 from here

In [ ]:
for entity in entities:
    if  entity.iri.startswith("http://purl.org/sig/ont/fma/fma"):
            # Generate a new IRI
            new_name= entity.name[3:]
            new_iri = f"{correct_prefix_fma}{new_name}"
            # Update the entity's IRI and assign the label
            try:
              entity.iri = new_iri
            except:
              print("Prefix change failed: ", entity.iri, new_iri)

In [ ]:
flap_onto.save(file="/content/drive/MyDrive/flap_onto_corrected_14.owl")

Flap onto corrected 14 from here

In [ ]:
flap_onto = get_ontology("file:///content/drive/MyDrive/flap_onto_corrected_14.owl").load()

In [ ]:
entities=[i for i in flap_onto.classes()]

In [ ]:
for entity in entities:
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0323716":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_24876a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0326497":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_25000a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0326530":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_24876a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0326542":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_63949a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0327063":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_55220a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0327086":
    entity.iri = "https://purl.bioontology.org/ontology/OFL/vein_from_gastrocnemius_muscle"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0327499":
    entity.iri = "https://purl.bioontology.org/ontology/OFL/artery_to_soleus_muscle"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0327608":
    entity.iri = "https://purl.bioontology.org/ontology/OFL/vein_to_soleus_muscle"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0329033":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_9712a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0329037":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_24979a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0329063":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_24967a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0329072":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_9663a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0329094":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_9664a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0329102":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_24890a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_0329137":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_24964a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_17739":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_17559a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_278104":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_12795a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_300146":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_9663b"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_300148":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_9663c"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_300612":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_34830a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_312822":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_25245a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_321487":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_24864a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_321491":
    entity.iri = "https://purl.bioontology.org/ontology/OFL/second_to_fifth_metacarpal_artery"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_321641":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_10561a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_321947":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_24040a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_322178":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_24219a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_322319":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_79743a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_322377":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_9666a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_322396":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_38125a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_322630":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_63884a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_322733":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_79759a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_323204":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_45205a"
  if entity.iri == "http://purl.obolibrary.org/obo/FMA_323208":
    entity.iri = "http://purl.obolibrary.org/obo/FMA_45205b"



In [ ]:
print([restriction for restriction in obo.FMA_0323716._equivalent_to])

[]


In [ ]:
flap_onto.save(file="/content/drive/MyDrive/flap_onto_corrected_15.owl")

Fixing owl class a546ihu  54i6j4 5o6ih

In [ ]:
flap_onto = get_ontology("file:///content/drive/MyDrive/flap_onto_corrected_13.owl").load()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/flap_onto_corrected_13.owl'

In [ ]:
flap_onto.save(file="/content/drive/MyDrive/flap_onto_corrected_16.owl")

In [ ]:
flap_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_16.owl").load()

In [ ]:
 query = """
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX owl: <http://www.w3.org/2002/07/owl#>
        PREFIX obo: <http://purl.obolibrary.org/obo/>
        PREFIX ofl: <https://purl.bioontology.org/ontology/OFL/>
        PREFIX fma: <http://purl.obolibrary.org/obo/FMA_>

SELECT DISTINCT ?fma_class
WHERE {
  # Finde alle explizit deklarierten Klassen
  { ?fma_class a owl:Class . }
  UNION
  # Finde auch Klassen, die als Typ für eine Instanz verwendet werden
  { [] a ?fma_class . }
  UNION
  # Finde auch Klassen, die in SubClassOf-Beziehungen vorkommen
  { [] rdfs:subClassOf ?fma_class . }
  UNION
  { ?fma_class rdfs:subClassOf [] . }

  # Filtere die Ergebnisse, sodass nur FMA-Klassen übrig bleiben
  FILTER(STRSTARTS(STR(?fma_class), "http://purl.obolibrary.org/obo/FMA_"))}"""


    # --- Step 3: Execute and display the results ---
results = list(flap_onto.world.sparql(query))


In [ ]:
print(results)

[[obo.FMA_10461], [obo.FMA_10462], [obo.FMA_10474], [obo.FMA_10561], [obo.FMA_10641], [obo.FMA_10646], [obo.FMA_10648], [obo.FMA_10649], [obo.FMA_10656], [obo.FMA_10657], [obo.FMA_10663], [obo.FMA_10664], [obo.FMA_11195], [obo.FMA_12165], [obo.FMA_12286], [obo.FMA_12528], [obo.FMA_12795], [obo.FMA_13109], [obo.FMA_13110], [obo.FMA_13303], [obo.FMA_13321], [obo.FMA_13324], [obo.FMA_13335], [obo.FMA_13342], [obo.FMA_13354], [obo.FMA_13357], [obo.FMA_13394], [obo.FMA_13397], [obo.FMA_13407], [obo.FMA_13891], [obo.FMA_13902], [obo.FMA_14319], [obo.FMA_14650], [obo.FMA_14781], [obo.FMA_14796], [obo.FMA_14808], [obo.FMA_15370], [obo.FMA_15390], [obo.FMA_15397], [obo.FMA_15402], [obo.FMA_15570], [obo.FMA_16482], [obo.FMA_16485], [obo.FMA_16487], [obo.FMA_16510], [obo.FMA_16511], [obo.FMA_16589], [obo.FMA_16914], [obo.FMA_18083], [obo.FMA_18868], [obo.FMA_18871], [obo.FMA_18908], [obo.FMA_18911], [obo.FMA_19841], [obo.FMA_19923], [obo.FMA_20110], [obo.FMA_20357], [obo.FMA_20358], [obo.FMA_2068

In [ ]:
output_filename = "fma_klassen_liste.txt"

with open(output_filename, "w") as file:
    # Loop through each row in the results
    for list_obj in results:
        entity=list_obj[0]
        # Each row is a tuple, e.g., (URIRef('...')), so we take the first element
        # Write the IRI followed by a newline character
        file.write(entity.iri + "\n")

print(f"Successfully wrote {len(results)} IRIs to {output_filename}")



Successfully wrote 420 IRIs to fma_klassen_liste.txt


In [ ]:
fma_subset=get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma_subset.owl").load()
flap_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_17.owl").load()

In [ ]:
entities=list(flap_onto.classes()) + list(flap_onto.object_properties()) + list(flap_onto.data_properties()) + list(flap_onto.individuals())

In [ ]:
# Precompile regex patterns once
fma_pattern = re.compile(r'\(FMA:\d+\)')
space_cleanup_pattern = re.compile(r'\s{2,}')

for entity in entities:
    if entity.label:
        for i, label in enumerate(entity.label):
            text = str(label)
            text= text[0].upper() + text[1:].lower()
            entity.label= [locstr(text, 'en')]

            if fma_pattern.search(text):
                cleaned_text = fma_pattern.sub('', text)
                cleaned_text = space_cleanup_pattern.sub(' ', cleaned_text).strip()
                if cleaned_text:
                    # Assuming locstr is defined elsewhere
                    entity.label= [locstr(cleaned_text, 'en')]
                    print(entity.label)
                    print(f"Original: {text}, Cleaned: {cleaned_text}")
                else:
                    print(f"Original: {text}, Cleaned: (empty string) — Skipped update")


In [ ]:
inferred_flap_onto = get_ontology("https://purl.bioontology.org/ontology/OFL_inferred/")

with inferred_flap_onto:
  sync_reasoner(infer_property_values = True)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/hermit:/usr/local/lib/python3.12/dist-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmp_l_za5hz -Y


* Owlready * Adding relation flap_onto_corrected_17.OFLID13291 OFLID13297 flap_onto_corrected_17.OFLID13288
* Owlready * Adding relation flap_onto_corrected_17.OFLID13294 OFLID13297 flap_onto_corrected_17.OFLID13288
* Owlready * Adding relation flap_onto_corrected_17.OFLID13307 OFLID13297 flap_onto_corrected_17.OFLID13288
* Owlready * Adding relation flap_onto_corrected_17.OFLID13299 OFLID13297 flap_onto_corrected_17.OFLID13300
* Owlready * Adding relation flap_onto_corrected_17.OFLID13292 OFLID13297 flap_onto_corrected_17.OFLID13288
* Owlready * Adding relation flap_onto_corrected_17.OFLID13293 OFLID13297 flap_onto_corrected_17.OFLID13288
* Owlready * Adding relation flap_onto_corrected_17.OFLID13298 OFLID13297 flap_onto_corrected_17.OFLID13300
* Owlready * Adding relation flap_onto_corrected_17.OFLID13288 OFLID13296 flap_onto_corrected_17.OFLID13294
* Owlready * Adding relation flap_onto_corrected_17.OFLID13288 OFLID13296 flap_onto_corrected_17.OFLID13307


* Owlready2 * HermiT took 6.341577768325806 seconds
* Owlready * Equivalenting: flap_onto_corrected_17.OFLID10026 flap_onto_corrected_17.OFLID10147
* Owlready * Equivalenting: flap_onto_corrected_17.OFLID10147 flap_onto_corrected_17.OFLID10026
* Owlready * Reparenting flap_onto_corrected_17.OFLID10029: {flap_onto_corrected_17.OFLID10030} => {obo.FMA_24876, flap_onto_corrected_17.OFLID10102, flap_onto_corrected_17.OFLID10030}
* Owlready * Reparenting flap_onto_corrected_17.OFLID10073: {flap_onto_corrected_17.OFLID10074} => {flap_onto_corrected_17.OFLID10005, flap_onto_corrected_17.OFLID10074}
* Owlready * Reparenting flap_onto_corrected_17.OFLID10043: {owl.Thing} => {flap_onto_corrected_17.OFLID10032, flap_onto_corrected_17.OFLID10112}
* Owlready * Reparenting flap_onto_corrected_17.OFLID10152: {owl.Thing} => {flap_onto_corrected_17.OFLID10112, flap_onto_corrected_17.OFLID10102}
* Owlready * Reparenting flap_onto_corrected_17.OFLID10040: {flap_onto_corrected_17.OFLID10041} => {flap_onto

In [ ]:
flap_onto.save(file="/content/drive/MyDrive/flap_onto_corrected_18.owl")
inferred_flap_onto.save(file="/content/drive/MyDrive/flap_onto_inferred_18.owl")

In [ ]:
with inferred_flap_onto:
  sync_reasoner(infer_property_values = True)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/hermit:/usr/local/lib/python3.12/dist-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmp666_7jz5 -Y
* Owlready2 * HermiT took 6.322809219360352 seconds
* Owlready * Equivalenting: flap_onto_corrected_17.OFLID10026 flap_onto_corrected_17.OFLID10147
* Owlready * Equivalenting: flap_onto_corrected_17.OFLID10147 flap_onto_corrected_17.OFLID10026
* Owlready * Reparenting flap_onto_corrected_17.OFLID1000060: {flap_onto_corrected_17.OFLID1000055, owl.ObjectProperty} => {flap_onto_corrected_17.OFLID1000055}
* Owlready * Reparenting flap_onto_corrected_17.OFLID1000056: {fma.nerve_supply_of, owl.ObjectProperty} => {fma.nerve_supply_of}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [ ]:
flap_onto.save(file="/content/drive/MyDrive/flap_onto_corrected_19.owl")
inferred_flap_onto.save(file="/content/drive/MyDrive/flap_onto_inferred_19.owl")

In [ ]:
flap_onto.destroy()

In [ ]:
fma_subset=get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma_subset.owl").load()
inferred_flap_onto= get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_inferred_19.owl").load()
flap_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_19.owl").load()

In [ ]:
flap_onto.save(file="/content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_20.owl")
inferred_flap_onto.save(file="/content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_inferred_20.owl")

In [ ]:
fma_subset=get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma_subset.owl").load()
flap_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_20.owl").load()

In [ ]:
for indi in list(flap_onto.individuals()):
        print(indi.label,indi.name)
        try:
          old_values = list(indi.part) + list(indi.BFO_0000051)
        except:
          try:
            print("indi.part copyied to old values")
            old_values = list(indi.part)
          except:
            try:
              print("indi.bfo copyied to old values")
              old_values = list(indi.BFO_0000051)
            except:
              print("No old values" )
              old_values=[]
        print(f"Old values of {indi.label} are {old_values}")
        if old_values:
            print("old_values added to indi.oflpart")
            indi.OFLID13296=old_values
            try:
              print("indi.part deleted")
              indi.part=[]
            except:
              print(f"No values for fma part for {indi.label}")
            try:
              print("indi.bfo deleted")
              indi.BFO_0000051=[]
            except:
              print(f"No values for bfo part for {indi.label}")
        else:
            print(f"No values at all for {indi.label}")

[locstr('Nicos alt lappen', 'en')] OFLID106032
Old values of [locstr('Nicos alt lappen', 'en')] are []
No values at all for [locstr('Nicos alt lappen', 'en')]
[locstr('Alt flap of patient x', 'en')] OFLID13288
Old values of [locstr('Alt flap of patient x', 'en')] are []
No values at all for [locstr('Alt flap of patient x', 'en')]
[locstr('Descending branch of left lateral circumflex femoral artery of patient x', 'en')] OFLID13294
Old values of [locstr('Descending branch of left lateral circumflex femoral artery of patient x', 'en')] are []
No values at all for [locstr('Descending branch of left lateral circumflex femoral artery of patient x', 'en')]
[locstr('Left lateral circumflex femoral vein of patient x', 'en')] OFLID13307
Old values of [locstr('Left lateral circumflex femoral vein of patient x', 'en')] are []
No values at all for [locstr('Left lateral circumflex femoral vein of patient x', 'en')]
[locstr('Flap harvest process of alt flap of patient x', 'en')] OFLID13298
Old values

In [ ]:
for indi in list(flap_onto.individuals()):
        print(indi.label,indi.name)
        try:
          old_values = list(indi.part_of) + list(indi.BFO_0000050)
        except:
          try:
            print("indi.part_of copyied to old values")
            old_values = list(indi.part_of)
          except:
            try:
              print("indi.bfo_part_of copyied to old values")
              old_values = list(indi.BFO_0000050)
            except:
              print("No old values" )
              old_values=[]
        print(f"Old values of {indi.label} are {old_values}")
        if old_values:
            print("old_values added to indi.oflpartof")
            indi.OFLID13297=old_values
            try:
              print("indi.part deleted")
              indi.part_of=[]
            except:
              print(f"No values for fma part of for {indi.label}")
            try:
              print("indi.bfo part of deleted")
              indi.BFO_0000050=[]
            except:
              print(f"No values for bfo part of for {indi.label}")
        else:
            print(f"No values at all for {indi.label}")

[locstr('Nicos alt lappen', 'en')] OFLID106032
indi.part_of copyied to old values
indi.bfo_part_of copyied to old values
No old values
Old values of [locstr('Nicos alt lappen', 'en')] are []
No values at all for [locstr('Nicos alt lappen', 'en')]
[locstr('Alt flap of patient x', 'en')] OFLID13288
indi.part_of copyied to old values
indi.bfo_part_of copyied to old values
No old values
Old values of [locstr('Alt flap of patient x', 'en')] are []
No values at all for [locstr('Alt flap of patient x', 'en')]
[locstr('Complete flap operation process of alt flap of patient x', 'en')] OFLID13300
indi.part_of copyied to old values
indi.bfo_part_of copyied to old values
No old values
Old values of [locstr('Complete flap operation process of alt flap of patient x', 'en')] are []
No values at all for [locstr('Complete flap operation process of alt flap of patient x', 'en')]
[locstr('Volume of alt flap of patient x', 'en')] OFLID13304
indi.part_of copyied to old values
indi.bfo_part_of copyied to ol

In [ ]:
#with inferred_flap_onto:
sync_reasoner(infer_property_values = True)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/hermit:/usr/local/lib/python3.12/dist-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmpfemxcrk8 -Y
* Owlready2 * HermiT took 5.283539056777954 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [ ]:
flap_onto.save(file="/content/drive/MyDrive/flap_onto_corrected_and_inferred_21.owl")
# inferred_flap_onto.save(file="/content/drive/MyDrive/flap_onto_inferred_21.owl")

In [ ]:
fma_subset=get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma_subset.owl").load()
flap_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_20.owl").load()

NameError: name 'get_ontology' is not defined

In [ ]:
print(flap_onto.OFLID13297)

flap_onto_corrected_20.OFLID13297


In [ ]:
fma_subset=get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma_subset.owl").load()
flap_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_20.owl").load()

In [ ]:
inferred_flap_onto = get_ontology("https://purl.bioontology.org/ontology/OFL_inferred/")

with inferred_flap_onto:
  sync_reasoner(infer_property_values = True)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/hermit:/usr/local/lib/python3.12/dist-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmp9evq77yx -Y
* Owlready2 * HermiT took 6.786550760269165 seconds
* Owlready * Reparenting flap_onto_corrected_20.OFLID13286: {owl.ObjectProperty, flap_onto_corrected_20.OFLID12001} => {flap_onto_corrected_20.OFLID12001}
* Owlready * Reparenting flap_onto_corrected_20.OFLID1000068: {fma.has_location, owl.TransitiveProperty, owl.ObjectProperty} => {fma.has_location, owl.TransitiveProperty}
* Owlready * Reparenting flap_onto_corrected_20.OFLID1000052: {flap_onto_corrected_20.OFLID1000049, owl.ObjectProperty} => {flap_onto_corrected_20.OFLID1000049}
* Owlready * Reparenting fma.part_of: {flap_onto_corrected_20.OFLID13297, owl.ObjectProperty} => {flap_onto_corrected_20.OFLID13297}
* Owlready * Reparenting obo.BFO_0000050: {flap_onto_corrected_20.OFL

In [ ]:
inferred_flap_onto.save(file="/content/drive/MyDrive/flap_onto_inferred_22.owl")

In [ ]:
flap_onto.destroy()

In [ ]:
flap_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_22.owl").load()
fma_subset=get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma_subset.owl").load()

In [ ]:
inferred_flap_onto = get_ontology("https://purl.bioontology.org/ontology/OFL_inferred/")

with inferred_flap_onto:
  sync_reasoner(infer_property_values = True)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/hermit:/usr/local/lib/python3.12/dist-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmpq5db9skk -Y
* Owlready2 * HermiT took 6.421141862869263 seconds
* Owlready * Reparenting flap_onto_corrected_22.OFLID13287: {flap_onto_corrected_22.OFLID12002, owl.ObjectProperty} => {flap_onto_corrected_22.OFLID12002}
* Owlready * Reparenting fma.part: {flap_onto_corrected_22.OFLID13296, owl.ObjectProperty} => {flap_onto_corrected_22.OFLID13296}
* Owlready * Reparenting obo.BFO_0000051: {flap_onto_corrected_22.OFLID13296, owl.ObjectProperty} => {flap_onto_corrected_22.OFLID13296}
* Owlready * Reparenting flap_onto_corrected_22.OFLID13285: {flap_onto_corrected_22.OFLID12006, owl.ObjectProperty} => {flap_onto_corrected_22.OFLID12006}
* Owlready * Reparenting flap_onto_corrected_22.OFLID1000067: {owl.TransitiveProperty, owl.ObjectProperty, fma.h

In [ ]:
inferred_flap_onto.save(file="/content/drive/MyDrive/flap_onto_inferred_22.owl")

In [ ]:
with flap_onto:
  sync_reasoner(infer_property_values = True)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/hermit:/usr/local/lib/python3.12/dist-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmp7fst91_9 -Y
* Owlready2 * HermiT took 7.031298875808716 seconds
* Owlready * Reparenting flap_onto_corrected_22.OFLID13285: {flap_onto_corrected_22.OFLID12006, owl.ObjectProperty} => {flap_onto_corrected_22.OFLID12006}
* Owlready * Reparenting flap_onto_corrected_22.OFLID13286: {flap_onto_corrected_22.OFLID12001, owl.ObjectProperty} => {flap_onto_corrected_22.OFLID12001}
* Owlready * Reparenting obo.BFO_0000051: {flap_onto_corrected_22.OFLID13296, owl.ObjectProperty} => {flap_onto_corrected_22.OFLID13296}
* Owlready * Reparenting fma.part: {flap_onto_corrected_22.OFLID13296, owl.ObjectProperty} => {flap_onto_corrected_22.OFLID13296}
* Owlready * Reparenting obo.BFO_0000050: {flap_onto_corrected_22.OFLID13297, owl.ObjectProperty} => {flap_onto_

In [ ]:
flap_onto.save(file="/content/drive/MyDrive/flap_onto_corrected_and_inferred_22.owl")

In [ ]:
flap_onto = get_ontology("/content/drive/MyDrive/flap_onto_corrected_and_inferred_22.owl").load()
fma_subset=get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma_subset.owl").load()

OwlReadyOntologyParsingError: Cannot download 'http://purl.obolibrary.org/obo/fma/releases/2020-04-13/fma.owl'!

In [ ]:
create_gephi_graph("/content/drive/MyDrive/flap_onto_corrected_and_inferred_22.owl","flap_onto_corrected_and_inferred_22")

In [ ]:
flap_onto = get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_and_inferred_23.owl").load()
fma_subset=get_ontology("file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/fma_subset.owl").load()

In [ ]:
g = rdflib.Graph()

# Load your ontology file into the graph.
# rdflib can parse OWL files directly.
# Replace with the actual path to your file.
ontology_file = "file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_and_inferred_23.owl"
print(f"Loading {ontology_file}...")
g.parse(ontology_file, format="xml") # Use "turtle" if your file is in Turtle format
print("...loading complete.")

sparql_query = """
    PREFIX fma: <http://purl.obolibrary.org/obo/FMA_>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT DISTINCT ?class
    WHERE {
      # Condition 1: The class must have at least one direct superclass from FMA
      ?class rdfs:subClassOf ?super_fma .
      FILTER(STRSTARTS(STR(?super_fma), STR(fma:)))

      # Condition 2: The class must NOT have any direct superclass that is NOT from FMA
      MINUS {
        ?class rdfs:subClassOf ?super_not_fma .
        FILTER(!STRSTARTS(STR(?super_not_fma), STR(fma:)))
      }
    }
    ORDER BY ?class
"""

print("\nExecuting SPARQL query...")
# Execute the query
results = g.query(sparql_query)
print(f"Found {len(results)} results.")

# Loop through the results and print them


Loading file:///content/drive/Othercomputers/Mein MacBook Pro/Flap ontology/flap_onto_corrected_and_inferred_23.owl...
...loading complete.

Executing SPARQL query...
Found 8 results.


In [ ]:
print("\nClasses that are solely extensions of FMA:")
for row in results:
    # The result 'row' is a tuple, access the 'class' variable by its name or index
    print("ofl:", flap_onto.world[str(row[0])].name, ", label:", flap_onto.world[str(row[0])].label[0])


Classes that are solely extensions of FMA:
ofl: OFLID1000040 , label: Ascending branch of circumflex scapular artery
ofl: OFLID10001 , label: Arterial plexus of the integument
ofl: OFLID10414 , label: Subcutaneous adipose tissue of subdivision of cardinal body part
ofl: OFLID106012 , label: Perforator vessel
ofl: OFLID11152 , label: Vein from gastrocnemius muscle
ofl: OFLID11153 , label: Artery to soleus muscle
ofl: OFLID11154 , label: Vein to soleus muscle
ofl: OFLID11155 , label: Second to fifth metacarpal artery


In [ ]:
sparql_query = """
PREFIX fma: <http://purl.obolibrary.org/obo/FMA_>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX ofl: <https://purl.bioontology.org/ontology/OFL/>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?class ?class_citation
WHERE {

  # These parts are MANDATORY
  ?class rdfs:subClassOf* ofl:OFLID10002 .
  # ?class rdfs:label ?class_label .
  ?class <http://purl.org/dc/terms/bibliographicCitation> ?class_citation

}
"""

print("\nExecuting SPARQL query...")
# Execute the query
results = g.query(sparql_query)
print(f"Found {len(results)} results.")

# Loop through the results and print them
for row in results:
  print(row)


Executing SPARQL query...
Found 6 results.
(rdflib.term.URIRef('https://purl.bioontology.org/ontology/OFL/OFLID10073'), rdflib.term.Literal('Arpaci, E., R. E. Unlu, S. Altun, und N. M. Ertas. „Super Kutler Flap: An Alternative Technique for Reconstruction of Fingertip Defects“. Journal of Hand Surgery (European Volume) 42, Nr. 6 (1. Juli 2017): 626–32. https://doi.org/10.1177/1753193416686884.', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.URIRef('https://purl.bioontology.org/ontology/OFL/OFLID10073'), rdflib.term.Literal('Chakraborty, Sourabh Shankar, Prakash Chandra Kala, Ranjit Kumar Sahu, Pawan Kumar Dixit, Deepti Katrolia, und Suresh Kotu. „Fingertip Amputation Reconstruction with VY Advancement Flap: Literature Review and Comparative Analysis of Atasoy and Kutler Flaps“. World Journal of Plastic Surgery 10, Nr. 3 (September 2021): 8–17. https://doi.org/10.29252/wjps.10.3.8.', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#s

In [ ]:
for flap_class in flap_onto.classes():
  print(flap_class[]) if flap_class.comment != [] else False

['Random flaps rely on blood supply from the subdermal plexus, which is supplied by unnamed musculocutaneous perforators.']
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggregate of the Fiat object 'Flap pedical'"]
["Is an object aggrega

In [ ]:
# mimicking SPARQL query
results_list = []
for flap_class in flap_onto.classes():
    # Get the label (owlready2 gets the first one by default)
    label = flap_class.label.first() if flap_class.label else "No Label"

    # Get the citation (it might not exist, so we handle that)
    # You need to find the actual property object for bibliographicCitation
    citation_prop = IRIS["http://purl.org/dc/terms/bibliographicCitation"]
    try:
      citations = flap_class[citation_prop] # Get all values for this property
      for citation in citations:
            results_list.append((str(flap_class.iri), label, str(citation)))
    except:
        # If no citation, append with None, just like SPARQL's OPTIONAL
        results_list.append((str(flap_class.iri), label, None))

# Sort the results by label for nice presentation
results_list.sort(key=lambda x: x[1])

# Print the final, sorted results
for iri, label, citation in results_list:
    print(f"Class: {iri}\n  Label: {label}\n  Citation: {citation}\n")

Class: http://purl.obolibrary.org/obo/FMA_9577
  Label: Abdomen
  Citation: None

Class: http://purl.obolibrary.org/obo/FMA_19841
  Label: Abdominal fascia
  Citation: None

Class: http://purl.obolibrary.org/obo/FMA_37451
  Label: Abductor digiti minimi of foot
  Citation: None

Class: http://purl.obolibrary.org/obo/FMA_37382
  Label: Abductor digiti minimi of hand
  Citation: None

Class: http://purl.obolibrary.org/obo/FMA_37448
  Label: Abductor hallucis
  Citation: None

Class: http://purl.obolibrary.org/obo/FMA_37373
  Label: Abductor pollicis brevis
  Citation: None

Class: http://purl.obolibrary.org/obo/FMA_20110
  Label: Adipose tissue
  Citation: None

Class: https://purl.bioontology.org/ontology/OFL/OFLID10070
  Label: Advancement flaps
  Citation: None

Class: http://purl.obolibrary.org/obo/FMA_3726
  Label: Anastomosis
  Citation: None

Class: http://purl.obolibrary.org/obo/FMA_62955
  Label: Anatomical entity
  Citation: None

Class: http://purl.obolibrary.org/obo/FMA_67135